In [38]:
import pandas as pd
import os
import numpy as np

In [39]:
CLEANED_PATH = "../datasets/cleaned"
MASTER_PATH = "../datasets/master"

os.makedirs(MASTER_PATH, exist_ok=True)

In [40]:
schools = pd.read_csv(
    os.path.join(CLEANED_PATH, "schools_2025_clean.csv")
)

libraries = pd.read_csv(
    os.path.join(CLEANED_PATH, "libraries_2025_clean.csv")
)

reading_fondness = pd.read_csv(
    os.path.join(CLEANED_PATH, "reading_fondness_2025_clean.csv")
)

completion = pd.read_csv(
    os.path.join(CLEANED_PATH, "completion_rate_2023_clean.csv")
)

facilities = pd.read_csv(
    os.path.join(CLEANED_PATH, "facilities_2025_clean.csv")
)

eys = pd.read_csv(
    os.path.join(CLEANED_PATH, "eys_2025_clean.csv")
)

education_25plus = pd.read_csv(
    os.path.join(CLEANED_PATH, "education_25plus_2025_clean.csv")
)

literacy = pd.read_csv(
    os.path.join(CLEANED_PATH, "literacy_2025_clean.csv")
)

In [41]:
datasets = {
    "schools": schools,
    "libraries": libraries,
    "reading_fondness": reading_fondness,
    "completion": completion,
    "facilities": facilities,
    "eys": eys,
    "education_25plus": education_25plus,
    "literacy": literacy
}

for name, df in datasets.items():
    print(
        f"{name:20} "
        f"shape={str(df.shape):10} "
        f"provinces={df['province'].nunique():2}"
    )

schools              shape=(38, 10)   provinces=38
libraries            shape=(38, 18)   provinces=38
reading_fondness     shape=(38, 5)    provinces=38
completion           shape=(38, 4)    provinces=38
facilities           shape=(38, 6)    provinces=38
eys                  shape=(38, 3)    provinces=38
education_25plus     shape=(38, 3)    provinces=38
literacy             shape=(38, 4)    provinces=38


In [42]:
master = schools.copy()

In [43]:
master = master.merge(
    libraries,
    on="province",
    how="left",
    validate="one_to_one"
)

In [44]:
master = master.merge(
    reading_fondness,
    on="province",
    how="left",
    validate="one_to_one"
)

master = master.merge(
    completion,
    on="province",
    how="left",
    validate="one_to_one"
)

master = master.merge(
    facilities,
    on="province",
    how="left",
    validate="one_to_one"
)

master = master.merge(
    eys,
    on="province",
    how="left",
    validate="one_to_one"
)

master = master.merge(
    education_25plus,
    on="province",
    how="left",
    validate="one_to_one"
)

master = master.merge(
    literacy,
    on="province",
    how="left",
    validate="one_to_one"
)

In [45]:
print(master.columns.tolist())

['province', 'schools_primary_public', 'schools_primary_private', 'schools_primary_total', 'teachers_primary_public', 'teachers_primary_private', 'teachers_primary_total', 'pupils_primary_public', 'pupils_primary_private', 'pupils_primary_total', 'special_library_a', 'special_library_b', 'special_library_c', 'special_library_total', 'school_library_a', 'school_library_b', 'school_library_c', 'school_library_total', 'academic_library_a', 'academic_library_b', 'academic_library_c', 'academic_library_total', 'public_library_a', 'public_library_b', 'public_library_c', 'public_library_total', 'public_library_all', 'reading_fondness_level', 'tgm_pre_reading', 'tgm_reading', 'tgm_post_reading', 'completion_elementary', 'completion_junior_high', 'completion_senior_high', 'villages_primary', 'villages_junior_high', 'villages_senior_high', 'villages_vocational', 'villages_university', 'eys_male', 'eys_female', 'education_25plus_male', 'education_25plus_female', 'literacy_male', 'literacy_female'

In [50]:
# Columns that should be numeric
cols_dash_as_zero = [
    'special_library_a', 'special_library_b', 'special_library_c',
    'special_library_total',
    'school_library_a', 'school_library_b', 'school_library_c',
    'school_library_total',
    'academic_library_a', 'academic_library_b', 'academic_library_c',
    'academic_library_total',
    'public_library_a', 'public_library_b', 'public_library_c',
    'public_library_total', 'public_library_all'
]

cols_dash_as_nan = [
    'tgm_pre_reading',
    'tgm_reading',
    'tgm_post_reading',
    'completion_elementary',
    'completion_junior_high',
    'completion_senior_high'
]

# Handle both '-' and '–'
dash_values = ['-', '–']

for c in cols_dash_as_zero:
    master[c] = (
        master[c]
        .replace(dash_values, '0')
        .astype(float)
    )

for c in cols_dash_as_nan:
    master[c] = (
        master[c]
        .replace(dash_values, np.nan)
        .astype(float)
    )

master['reading_fondness_level'] = (
    master['reading_fondness_level']
    .replace(dash_values, np.nan)
)

print("Missing after cleaning:")
print(master.isnull().sum()[master.isnull().sum() > 0])

print("\nShape:")
print(master.shape)

print("\nDtypes:")
print(master.dtypes.value_counts())

Missing after cleaning:
reading_fondness_level    5
tgm_pre_reading           5
tgm_reading               5
tgm_post_reading          5
completion_elementary     4
completion_junior_high    4
completion_senior_high    4
dtype: int64

Shape:
(38, 46)

Dtypes:
float64    40
int64       5
object      1
Name: count, dtype: int64


In [54]:
for c in [
    'reading_fondness_level',
    'tgm_pre_reading',
    'tgm_reading',
    'tgm_post_reading',
    'completion_elementary',
    'completion_junior_high',
    'completion_senior_high',
    'special_library_a'
]:
    vals = master[c].unique()
    weird = [v for v in vals if v in ['-', '–']]
    print(c, weird)

reading_fondness_level []
tgm_pre_reading []
tgm_reading []
tgm_post_reading []
completion_elementary []
completion_junior_high []
completion_senior_high []
special_library_a []


In [57]:
for c in ['completion_elementary', 'completion_junior_high', 'completion_senior_high']:
    print(c, master.loc[master[c] == '-', 'province'].tolist())

print()

for c in ['reading_fondness_level', 'tgm_pre_reading', 'tgm_reading', 'tgm_post_reading']:
    print(c, master.loc[master[c] == '–', 'province'].tolist())

completion_elementary []
completion_junior_high []
completion_senior_high []

reading_fondness_level []
tgm_pre_reading []
tgm_reading []
tgm_post_reading []


In [58]:
# Library counts: '–' = genuinely zero facilities of that type -> 0
cols_dash_as_zero = [
    'special_library_a', 'special_library_b', 'special_library_c', 'special_library_total',
    'school_library_a', 'school_library_b', 'school_library_c', 'school_library_total',
    'academic_library_a', 'academic_library_b', 'academic_library_c', 'academic_library_total',
    'public_library_a', 'public_library_b', 'public_library_c', 'public_library_total',
    'public_library_all'
]

# Rates/scores: '-' or '–' = data not available (survey not yet conducted) -> NaN
cols_dash_as_nan = [
    'reading_fondness_level',
    'tgm_pre_reading',
    'tgm_reading',
    'tgm_post_reading',
    'completion_elementary',
    'completion_junior_high',
    'completion_senior_high'
]

for c in cols_dash_as_zero:
    master[c] = master[c].replace('–', '0').astype(float)

for c in cols_dash_as_nan:
    master[c] = master[c].replace({'–': np.nan, '-': np.nan}).astype(float)

print("Missing values after cleaning:")
print(master.isnull().sum()[master.isnull().sum() > 0])

print("\nRows with any missing outcome data:")
print(
    master.loc[
        master[cols_dash_as_nan].isnull().any(axis=1),
        ['province'] + cols_dash_as_nan
    ]
)

Missing values after cleaning:
reading_fondness_level    5
tgm_pre_reading           5
tgm_reading               5
tgm_post_reading          5
completion_elementary     4
completion_junior_high    4
completion_senior_high    4
dtype: int64

Rows with any missing outcome data:
            province  reading_fondness_level  tgm_pre_reading  tgm_reading  \
16              BALI                     NaN              NaN          NaN   
33  PAPUA BARAT DAYA                     NaN              NaN          NaN   
35     PAPUA SELATAN                     NaN              NaN          NaN   
36      PAPUA TENGAH                     NaN              NaN          NaN   
37  PAPUA PEGUNUNGAN                     NaN              NaN          NaN   

    tgm_post_reading  completion_elementary  completion_junior_high  \
16               NaN                  98.43                   93.03   
33               NaN                    NaN                     NaN   
35               NaN                    N

In [59]:
MASTER_FILE = os.path.join(
    MASTER_PATH,
    "master_clean_2025.csv"
)

master.to_csv(
    MASTER_FILE,
    index=False
)

print("Saved:", MASTER_FILE)

Saved: ../datasets/master\master_clean_2025.csv


In [61]:
df = pd.read_csv('../datasets/master/master_clean_2025.csv')
 
# --- Region mapping ---
region_map = {
    'ACEH':'Sumatera','SUMATERA UTARA':'Sumatera','SUMATERA BARAT':'Sumatera','RIAU':'Sumatera',
    'JAMBI':'Sumatera','SUMATERA SELATAN':'Sumatera','BENGKULU':'Sumatera','LAMPUNG':'Sumatera',
    'KEP. BANGKA BELITUNG':'Sumatera','KEP. RIAU':'Sumatera',
    'DKI JAKARTA':'Jawa','JAWA BARAT':'Jawa','JAWA TENGAH':'Jawa','DI YOGYAKARTA':'Jawa',
    'JAWA TIMUR':'Jawa','BANTEN':'Jawa',
    'BALI':'Bali-Nusra','NUSA TENGGARA BARAT':'Bali-Nusra','NUSA TENGGARA TIMUR':'Bali-Nusra',
    'KALIMANTAN BARAT':'Kalimantan','KALIMANTAN TENGAH':'Kalimantan','KALIMANTAN SELATAN':'Kalimantan',
    'KALIMANTAN TIMUR':'Kalimantan','KALIMANTAN UTARA':'Kalimantan',
    'SULAWESI UTARA':'Sulawesi','SULAWESI TENGAH':'Sulawesi','SULAWESI SELATAN':'Sulawesi',
    'SULAWESI TENGGARA':'Sulawesi','GORONTALO':'Sulawesi','SULAWESI BARAT':'Sulawesi',
    'MALUKU':'Maluku-Papua','MALUKU UTARA':'Maluku-Papua','PAPUA BARAT':'Maluku-Papua',
    'PAPUA BARAT DAYA':'Maluku-Papua','PAPUA':'Maluku-Papua','PAPUA SELATAN':'Maluku-Papua',
    'PAPUA TENGAH':'Maluku-Papua','PAPUA PEGUNUNGAN':'Maluku-Papua',
}
df['region'] = df['province'].map(region_map)
assert df['region'].isnull().sum()==0, df.loc[df['region'].isnull(),'province']

In [62]:
# --- Derived indicators ---
# Facility drop-off ratio: how much secondary access lags behind primary access
df['facility_ratio_jhs'] = df['villages_junior_high'] / df['villages_primary']
df['facility_ratio_shs'] = df['villages_senior_high'] / df['villages_primary']
df['facility_ratio_vocational'] = df['villages_vocational'] / df['villages_primary']
df['facility_ratio_univ'] = df['villages_university'] / df['villages_primary']

In [63]:
# School density (schools per 1000 pupils) - infra intensity
df['school_density_primary'] = df['schools_primary_total'] / df['pupils_primary_total'] * 1000
df['pupil_teacher_ratio'] = df['pupils_primary_total'] / df['teachers_primary_total']

In [64]:
# Gender gaps (positive = female higher)
df['gap_eys'] = df['eys_female'] - df['eys_male']
df['gap_literacy'] = df['literacy_female'] - df['literacy_male']
df['gap_edu25plus'] = df['education_25plus_female'] - df['education_25plus_male']

In [65]:
# Library access composite (per 1000 pupils, since bigger provinces naturally have more)
df['library_total_all'] = df['school_library_total'] + df['public_library_all'] + df['academic_library_total'] + df['special_library_total']
df['library_per_1000pupils'] = df['library_total_all'] / df['pupils_primary_total'] * 1000

In [66]:
# Outcome composite
df['outcome_composite'] = df[['completion_senior_high','literacy_total']].mean(axis=1)
# average EYS
df['eys_avg'] = (df['eys_male']+df['eys_female'])/2

In [68]:
df.to_csv('../datasets/master/master_derived_2025.csv', index=False)
print(df[['province','region','facility_ratio_jhs','facility_ratio_shs','gap_eys','gap_literacy',
        'library_per_1000pupils','outcome_composite']].round(2).to_string())

                province        region  facility_ratio_jhs  facility_ratio_shs  gap_eys  gap_literacy  library_per_1000pupils  outcome_composite
0                   ACEH      Sumatera                0.43                0.22     0.32         -1.03                    0.82              86.69
1         SUMATERA UTARA      Sumatera                0.47                0.23     0.40         -0.78                    0.29              86.72
2         SUMATERA BARAT      Sumatera                0.65                0.34     0.98         -0.72                    1.26              83.86
3                   RIAU      Sumatera                0.67                0.35     0.37         -0.96                    0.69              83.40
4                  JAMBI      Sumatera                0.55                0.27     0.48         -2.28                    0.66              82.10
5       SUMATERA SELATAN      Sumatera                0.47                0.24     0.39         -0.94                    0.90     

In [69]:
df = pd.read_csv('../datasets/master/master_derived_2025.csv')

# Flag completeness
df['has_completion_data'] = df['completion_senior_high'].notnull()
df['has_reading_data'] = df['reading_fondness_level'].notnull()

In [70]:
# --- Composite ACCESS index (z-score average) ---
# Using indicators available for ALL 38 provinces
access_vars = ['facility_ratio_jhs','facility_ratio_shs','school_density_primary']
for v in access_vars:
    df[f'z_{v}'] = (df[v]-df[v].mean())/df[v].std()
df['access_index'] = df[[f'z_{v}' for v in access_vars]].mean(axis=1)

In [71]:
# --- Composite OUTCOME index ---
# Version A (n=38): literacy only (available for all)
df['z_literacy'] = (df['literacy_total']-df['literacy_total'].mean())/df['literacy_total'].std()
df['z_eys'] = (df['eys_avg']-df['eys_avg'].mean())/df['eys_avg'].std()
df['outcome_index_all'] = df[['z_literacy','z_eys']].mean(axis=1)

In [72]:
# Version B (n=34, has completion data): literacy + completion_shs + eys
sub = df[df['has_completion_data']].copy()
for v in ['completion_senior_high']:
    sub[f'z_{v}'] = (sub[v]-sub[v].mean())/sub[v].std()
sub['outcome_index_full'] = sub[['z_literacy','z_eys','z_completion_senior_high']].mean(axis=1)
df = df.merge(sub[['province','outcome_index_full']], on='province', how='left')

In [73]:
# --- Access-Outcome GAP (using full outcome index where available, else outcome_index_all) ---
df['outcome_index_used'] = df['outcome_index_full'].fillna(df['outcome_index_all'])
df['access_outcome_gap'] = df['access_index'] - df['outcome_index_used']
# Positive gap = access GOOD but outcome BAD relative to peers (quality problem)
# Negative gap = access BAD but outcome relatively OK, OR both low (need to check quadrant)

In [74]:
df.to_csv('../datasets/master/master_final_2025.csv', index=False)

In [75]:
print("=== TOP 5 BIGGEST GAP (access jauh lebih tinggi drpd outcome -> problem KUALITAS) ===")
print(df.nlargest(5,'access_outcome_gap')[['province','region','access_index','outcome_index_used','access_outcome_gap']].round(2).to_string())
 
print("\n=== TOP 5 GAP TERBALIK (outcome lebih tinggi drpd akses -> efisien walau akses terbatas) ===")
print(df.nsmallest(5,'access_outcome_gap')[['province','region','access_index','outcome_index_used','access_outcome_gap']].round(2).to_string())
 
print("\n=== Data completeness flags ===")
print(df[['province','has_completion_data','has_reading_data']].to_string())

=== TOP 5 BIGGEST GAP (access jauh lebih tinggi drpd outcome -> problem KUALITAS) ===
               province        region  access_index  outcome_index_used  access_outcome_gap
37     PAPUA PEGUNUNGAN  Maluku-Papua         -1.27               -4.21                2.94
36         PAPUA TENGAH  Maluku-Papua         -1.35               -2.81                1.46
17  NUSA TENGGARA BARAT    Bali-Nusra          1.14               -0.18                1.32
29       SULAWESI BARAT      Sulawesi          0.72               -0.49                1.21
18  NUSA TENGGARA TIMUR    Bali-Nusra          0.08               -0.70                0.78

=== TOP 5 GAP TERBALIK (outcome lebih tinggi drpd akses -> efisien walau akses terbatas) ===
            province        region  access_index  outcome_index_used  access_outcome_gap
0               ACEH      Sumatera         -0.26                0.78               -1.04
22  KALIMANTAN TIMUR    Kalimantan         -0.37                0.63               -0.99
1